In [1]:
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor, LGBMClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from tabpfn import TabPFNRegressor, TabPFNClassifier
from tabicl import TabICLRegressor, TabICLClassifier
from tabpfn_extensions.hpo import TunedTabPFNRegressor, TunedTabPFNClassifier
import matplotlib.pyplot as plt
from econml.metalearners import SLearner, TLearner, XLearner
from econml.dml import NonParamDML, CausalForestDML
from econml.dr import DRLearner
from sklearn.metrics import mean_squared_error
import warnings
import torch
import tabpfn
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold, KFold
from sklearn.metrics import mean_squared_error
try:
    from causalpfn import CATEEstimator
except ImportError:
    print("CausalPFN not installed. Please install with: pip install causalpfn")
print(tabpfn.__version__)
import time
from scipy.special import expit

from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold, KFold
from sklearn.metrics import mean_squared_error
import numpy as np

# Detect device
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print(f"Using device: {device}")

# CausalPFN does not support MPS
causalpfn_device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"CausalPFN device: {causalpfn_device}")

# TabICL does not support MPS
tabicl_device = "cpu" if device == "mps" else device
print(f"TabICL device: {tabicl_device}")

HPO_N_TRIALS = 20  # Bayesian HPO trials for TunedTabPFN
NUISANCE_CV  = 5   # K-fold cross-fitting for R- and DR-learner

# Set random seed for reproducibility
np.random.seed(42)

# Suppress warnings
warnings.filterwarnings("ignore")

import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/miniconda3/lib/python3.13/site-packages/hyperopt/atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


6.2.0
Using device: mps
CausalPFN device: cpu
TabICL device: cpu


In [2]:
IMBALANCE_LEVELS = [0.5, 0.3, 0.15, 0.05]
CONFOUNDING_ALPHAS = [0.0, 0.5, 1.0, 3.0, 100]

LGBM_GRID = {
    "num_leaves": [15, 31, 63],
    "min_child_samples": [20, 50, 100],
    "learning_rate": [0.01, 0.05, 0.1],
    "n_estimators": [200, 500, 1000],
}

N_ITER = 30
N_EST = 1000
NUISANCE_CV = 5

In [3]:

# ── Core tuner (pooled data — for S, R, DR learners) ─────────────────────────
def _tune_reg(X, y, stratify, grid, n_iter=N_ITER, seed=42):
    """Tune LGBMRegressor via CV. Returns best params dict."""
    X = np.asarray(X)
    base = LGBMRegressor(n_estimators=N_EST, random_state=seed, verbose=-1)
    cv = list(StratifiedKFold(n_splits=3, shuffle=True, random_state=seed).split(X, stratify))
    Search = RandomizedSearchCV if n_iter else GridSearchCV
    kw = dict(n_iter=n_iter, random_state=seed) if n_iter else {}
    search = Search(base, grid, scoring='neg_mean_squared_error', cv=cv, n_jobs=-1, **kw)
    search.fit(X, y)
    return search.best_params_

def _tune_cls(X, t, grid, n_iter=N_ITER, seed=42):
    """Tune LGBMClassifier via CV. Returns best params dict."""
    X = np.asarray(X)
    base = LGBMClassifier(n_estimators=N_EST, random_state=seed, verbose=-1)
    cv = list(StratifiedKFold(n_splits=3, shuffle=True, random_state=seed).split(X, t))
    Search = RandomizedSearchCV if n_iter else GridSearchCV
    kw = dict(n_iter=n_iter, random_state=seed) if n_iter else {}
    search = Search(base, grid, scoring='neg_log_loss', cv=cv, n_jobs=-1, **kw)
    search.fit(X, t)
    return search.best_params_

def _tune_reg_arm(X_arm, y_arm, grid, n_iter=N_ITER, seed=42):
    """Tune LGBMRegressor on a single treatment arm. Returns best params dict."""
    X_arm = np.asarray(X_arm)
    n_splits = min(3, max(2, len(y_arm) // 20))
    base = LGBMRegressor(n_estimators=N_EST, random_state=seed, verbose=-1)
    cv = list(KFold(n_splits=n_splits, shuffle=True, random_state=seed).split(X_arm))
    Search = RandomizedSearchCV if n_iter else GridSearchCV
    kw = dict(n_iter=n_iter, random_state=seed) if n_iter else {}
    search = Search(base, grid, scoring='neg_mean_squared_error', cv=cv, n_jobs=-1, **kw)
    search.fit(X_arm, y_arm)
    return search.best_params_


from econml.utilities import WeightedModelWrapper

def make_lgbm_final(seed=42):
    """Fresh RandomizedSearchCV for final stage — tuned on pseudo-outcomes at fit time."""
    return RandomizedSearchCV(
        LGBMRegressor(n_estimators=N_EST, random_state=seed, verbose=-1),
        LGBM_GRID, n_iter=N_ITER, cv=3, scoring='neg_mean_squared_error',
        n_jobs=-1, random_state=seed,
    )

def make_tabpfn_final():
    """WeightedModelWrapper lets TabPFN accept sample_weight in R-learner final stage."""
    return WeightedModelWrapper(TabPFNRegressor(device=device))

def make_tabicl_final():
    """WeightedModelWrapper lets TabICL accept sample_weight in R-learner final stage."""#
    return WeightedModelWrapper(TabICLRegressor(device=tabicl_device, random_state=42, verbose=False))#



def make_reg(params, seed=42):
    return LGBMRegressor(random_state=seed, verbose=-1, **params)

def make_cls(params, seed=42):
    return LGBMClassifier(random_state=seed, verbose=-1, **params)


# ── Metric functions ──────────────────────────────────────────────────────────
def calculate_pehe(predicted_ite, true_ite):
    return np.sqrt(mean_squared_error(true_ite, predicted_ite))

def calculate_ate_error(predicted_ite, true_ite):
    return np.abs(predicted_ite.mean() - true_ite.mean())

SENS_MODELS = [
    'S-LinearRegression', 'S-LightGBM', 'S-TabPFN', 'S-TabICL',
    'T-LinearRegression', 'T-LightGBM', 'T-TabPFN', 'T-TabICL',
    'X-LinearRegression', 'X-LightGBM', 'X-TabPFN', 'X-TabICL',
    'R-LinearRegression', 'R-LightGBM', 'R-TabPFN', 'R-TabICL',
    'DR-LinearRegression', 'DR-LightGBM', 'DR-TabPFN', 'DR-TabICL',
    'CausalForest',
    'CausalPFN',
]

In [4]:

# ── Helpers ───────────────────────────────────────────────────────────────────
def _lin_pred(X):
    """Fixed linear score propensity signal."""
    arr = X.values[:, :26] if isinstance(X, pd.DataFrame) else X[:, :25]
    weights = np.array([
         1.0,  0.5, -0.5,  0.3, -0.3,
         0.4, -0.4,  0.2, -0.2,  0.6,
        -0.6,  0.1, -0.1,  0.35,-0.35,
         0.45,-0.45, 0.25,-0.25,  0.55,
        -0.55, 0.15,-0.15, 0.65,-0.65,
    ])
    return arr @ weights


In [5]:
train_data = np.load('ihdp_npci_1-100.train.npz')
test_data  = np.load('ihdp_npci_1-100.test.npz')

_n_datasets   = 5
feature_names = [f"x{j}" for j in range(25)]

processed_datasets = []
for i in range(_n_datasets):
    X_train   = pd.DataFrame(train_data['x'][:, :, i], columns=feature_names)
    X_test    = pd.DataFrame(test_data['x'][:, :, i],  columns=feature_names)
    mu0_train = train_data['mu0'][:, i]
    mu1_train = train_data['mu1'][:, i]
    mu0_test  = test_data['mu0'][:, i]
    mu1_test  = test_data['mu1'][:, i]

    processed_datasets.append({
        'id':            i + 1,
        'X_train':       X_train,
        'X_test':        X_test,
        'mu0_train':     mu0_train,
        'mu1_train':     mu1_train,
        'true_ITE_test': mu1_test - mu0_test,
    })

print(f"Loaded {len(processed_datasets)} IHDP replications.")
print(f"Train size: {len(X_train)}, Test size: {len(X_test)}, Features: {X_train.shape[1]}")

Loaded 5 IHDP replications.
Train size: 672, Test size: 75, Features: 25


In [6]:


def _make_learner(name, s_s, r_nuisance, s_cls, s_ctrl, s_trt):
    # ── S-learners ────────────────────────────────────────────────────────────
    if name == 'S-LinearRegression':
        return SLearner(overall_model=LinearRegression())
    elif name == 'S-LightGBM':
        return SLearner(overall_model=make_reg(s_s, 42))
    elif name == 'S-TabPFN':
        return SLearner(overall_model=TabPFNRegressor(device=device))
    elif name == 'S-TabICL':
        return SLearner(overall_model=TabICLRegressor(device=tabicl_device, random_state=42, verbose=False))

    # ── T-learners ────────────────────────────────────────────────────────────
    elif name == 'T-LinearRegression':
        return TLearner(models=(LinearRegression(), LinearRegression()))
    elif name == 'T-LightGBM':
        return TLearner(models=(make_reg(s_ctrl, 42), make_reg(s_trt, 43)))
    elif name == 'T-TabPFN':
        return TLearner(models=(TabPFNRegressor(device=device), TabPFNRegressor(device=device)))
    elif name == 'T-TabICL':
        return TLearner(models=(
            TabICLRegressor(device=tabicl_device, random_state=42, verbose=False),
            TabICLRegressor(device=tabicl_device, random_state=43, verbose=False)))

    # ── X-learners ────────────────────────────────────────────────────────────
    elif name == 'X-LinearRegression':
        return XLearner(
            models=(LinearRegression(), LinearRegression()),
            cate_models=(LinearRegression(), LinearRegression()),
            propensity_model=LogisticRegression(max_iter=1000, random_state=42))
    elif name == 'X-LightGBM':
        return XLearner(
            models=(make_reg(s_ctrl, 42), make_reg(s_trt, 43)),
            cate_models=(make_lgbm_final(46), make_lgbm_final(47)),
            propensity_model=make_cls(s_cls, 44))
    elif name == 'X-TabPFN':
        return XLearner(
            models=(TabPFNRegressor(device=device), TabPFNRegressor(device=device)),
            cate_models=(TabPFNRegressor(device=device), TabPFNRegressor(device=device)),
            propensity_model=TabPFNClassifier(device=device))
    elif name == 'X-TabICL':
        return XLearner(
            models=(TabICLRegressor(device=tabicl_device, random_state=42, verbose=False),
                    TabICLRegressor(device=tabicl_device, random_state=43, verbose=False)),
            cate_models=(TabICLRegressor(device=tabicl_device, random_state=44, verbose=False),
                         TabICLRegressor(device=tabicl_device, random_state=45, verbose=False)),
            propensity_model=TabICLClassifier(device=tabicl_device, random_state=42, verbose=False))

    # ── R-learners ────────────────────────────────────────────────────────────
    elif name == 'R-LinearRegression':
        return NonParamDML(
            model_y=LinearRegression(),
            model_t=LogisticRegression(max_iter=1000, random_state=42),
            model_final=LinearRegression(),
            discrete_treatment=True, cv=NUISANCE_CV)
    elif name == 'R-LightGBM':
        return NonParamDML(
            model_y=make_reg(r_nuisance, 42),    # r_nuisance, tuned on X only
            model_t=make_cls(s_cls, 43),          # seed 43 to match original
            model_final=make_lgbm_final(44),      # original uses RandomizedSearchCV final, not fixed params
            discrete_treatment=True, cv=NUISANCE_CV)
    elif name == 'R-TabPFN':
        return NonParamDML(
            model_y=TabPFNRegressor(device=device),
            model_t=TabPFNClassifier(device=device),
            model_final=make_lgbm_final(44),
            discrete_treatment=True, cv=NUISANCE_CV)
    elif name == 'R-TabICL':
        return NonParamDML(
            model_y=TabICLRegressor(device=tabicl_device, random_state=42, verbose=False),
            model_t=TabICLClassifier(device=tabicl_device, random_state=42, verbose=False),
            model_final=make_lgbm_final(44),
            discrete_treatment=True, cv=NUISANCE_CV)

    # ── DR-learners ───────────────────────────────────────────────────────────
    elif name == 'DR-LinearRegression':
        return DRLearner(
            model_regression=LinearRegression(),
            model_propensity=LogisticRegression(max_iter=1000, random_state=42),
            model_final=LinearRegression(),
            min_propensity=0.05, cv=NUISANCE_CV)
    elif name == 'DR-LightGBM':
        return DRLearner(
            model_regression=make_reg(r_nuisance, 42),  # r_nuisance, tuned on X only
            model_propensity=make_cls(s_cls, 44),
            model_final=make_lgbm_final(45),              # RandomizedSearchCV final, same as original
            min_propensity=0.05, cv=NUISANCE_CV)
    elif name == 'DR-TabPFN':
        return DRLearner(
            model_regression=TabPFNRegressor(device=device),
            model_propensity=TabPFNClassifier(device=device),
            model_final=TabPFNRegressor(device=device),
            min_propensity=0.05, cv=NUISANCE_CV)
    elif name == 'DR-TabICL':
        return DRLearner(
            model_regression=TabICLRegressor(device=tabicl_device, random_state=42, verbose=False),
            model_propensity=TabICLClassifier(device=tabicl_device, random_state=42, verbose=False),
            model_final=TabICLRegressor(device=tabicl_device, random_state=42, verbose=False),
            min_propensity=0.05, cv=NUISANCE_CV)

    # ── CausalForest ──────────────────────────────────────────────────────────
    elif name == 'CausalForest':
        return CausalForestDML(
            model_y=make_reg(r_nuisance, 42),  # r_nuisance, not s_reg
            model_t=make_cls(s_cls, 43),        # seed 43 to match original
            discrete_treatment=True,
            n_estimators=200,                   # original uses 200, not 500
            min_samples_leaf=5,                 # original has this
            random_state=42)

    raise ValueError(f"Unknown model: {name}")


In [7]:
# ── Scenario generators ───────────────────────────────────────────────────────
def gen_randomized(X, mu0, mu1, rng, p=0.5):
    """
    Axis 1 / 2: No confounding.
    T ~ Bern(p), independent of X.  P(T|X) = P(T) = p.
    Y = T·mu1 + (1-T)·mu0.
    """
    T = rng.binomial(1, p, len(X))
    Y = T * mu1 + (1 - T) * mu0
    return T, Y


def gen_confounded(X, mu0, mu1, rng, alpha=1.0):
    """
    Axis 3: Confounding with controlled imbalance.
    P(T=1|X) = sigma(alpha · f̃(X)),  f̃(X) = f(X) − mean(f(X)).
    Centering ensures E[P(T=1|X)] ≈ 0.5 for all alpha,
    so imbalance and confounding are decoupled.
    """
    lp = _lin_pred(X)
    lp = (lp - lp.mean()) / (lp.std() + 1e-8)   # standardize: E[P(T)]=0.5, scale consistent across datasets
    prop = expit(alpha * lp)
    T = rng.binomial(1, prop, len(X))
    Y = T * mu1 + (1 - T) * mu0
    return T, Y


# ── Core evaluation loop ──────────────────────────────────────────────────────
def run_sensitivity_analysis(datasets, models=SENS_MODELS, n_reps=1, seed=42):
    """
    Two-axis sensitivity analysis over all ACIC 2016 instances.

    For each instance × axis × parameter × random seed:
      - Re-generates T (and Y) under the given scenario
      - Trains a meta-learner on (X_train, T_train, Y_train)
      - Evaluates PEHE and ATE error on (X_test, true_ITE_test)

    The ground truth true_ITE_test = mu1_test − mu0_test is the same
    in every scenario — only the training difficulty changes.

    Parameters
    ----------
    datasets : list of dicts with keys:
        X_train, X_test, mu0_train, mu1_train, true_ITE_test
    models   : list of str — model names from SENS_MODELS
    n_reps   : int — random T-assignment seeds per instance per scenario
    seed     : int — master random seed

    Returns
    -------
    pd.DataFrame with columns:
        axis, param, model, rep, seed_rep, pehe, ate_error
    """
    rng_master = np.random.default_rng(seed)
    rows = []
    timings = {m: [] for m in models}  # elapsed seconds per fit, per model

    # Resolve CausalPFN device once — avoids repeated torch calls inside inner loop
    _cpfn_device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    for ds in datasets:
        rep_id = ds['id']
        X_tr, X_te     = ds['X_train'], ds['X_test']
        mu0_tr, mu1_tr = ds['mu0_train'], ds['mu1_train']
        true_ite       = ds['true_ITE_test']
        print(f"\n── Rep {rep_id:3d}/{len(datasets)} ────────────────────────────────")

        for seed_rep in range(n_reps):
            rng = np.random.default_rng(rng_master.integers(0, int(1e9)))

            # ── Axis 2: imbalance ─────────────────────────────────────────────
            for p in IMBALANCE_LEVELS:
                T_tr, Y_tr = gen_randomized(X_tr, mu0_tr, mu1_tr, rng, p=p)
                if T_tr.sum() < 5 or (1 - T_tr).sum() < 5:
                    continue   # degenerate split — skip

                # Tune LightGBM on THIS scenario's data — best model for this exact setting
                t0 = time.time()
                ctrl_mask  = T_tr == 0
                trt_mask   = T_tr == 1
                X_with_T   = np.column_stack([X_tr, T_tr])
                s_s        = _tune_reg(X_with_T,         Y_tr, T_tr, LGBM_GRID, N_ITER)
                r_nuisance = _tune_reg(X_tr,             Y_tr, T_tr, LGBM_GRID, N_ITER)
                s_cls      = _tune_cls(X_tr,             T_tr,       LGBM_GRID, N_ITER)
                s_ctrl     = _tune_reg_arm(X_tr[ctrl_mask], Y_tr[ctrl_mask], LGBM_GRID, N_ITER)
                s_trt      = _tune_reg_arm(X_tr[trt_mask],  Y_tr[trt_mask],  LGBM_GRID, N_ITER)
                print(f"  LightGBM tuning: {time.time() - t0:.1f}s")

                for mname in models:
                    try:
                        t0 = time.time()
                        if mname == 'CausalPFN':
                            learner = CATEEstimator(device=_cpfn_device, verbose=False)
                            learner.fit(
                                np.asarray(X_tr, dtype=np.float32),
                                T_tr.astype(np.float32),
                                Y_tr.astype(np.float32))
                            te = np.asarray(
                                learner.estimate_cate(np.asarray(X_te, dtype=np.float32))
                            ).reshape(-1)
                        else:
                            learner = _make_learner(mname, s_s, r_nuisance, s_cls, s_ctrl, s_trt)
                            if mname == 'CausalForest':
                                learner.tune(Y_tr, T_tr, X=X_tr)   # original calls .tune() before .fit()
                            learner.fit(Y_tr, T_tr, X=X_tr)
                            te = learner.effect(X_te)
                        elapsed = time.time() - t0
                        timings[mname].append(elapsed)
                        print(f"  imb p={p:.2f} | {mname:20s} | {elapsed:.3f}s")
                        rows.append(dict(
                            axis='imbalance', param=p, model=mname,
                            rep=rep_id, seed_rep=seed_rep,
                            pehe=calculate_pehe(te, true_ite),
                            ate_error=calculate_ate_error(te, true_ite),
                        ))
                    except Exception as e:
                        print(f"\n[imbalance] {mname} rep={rep_id} p={p}: {e}")

            # ── Axis 3: confounding ───────────────────────────────────────────
            for alpha in CONFOUNDING_ALPHAS:
                T_tr, Y_tr = gen_confounded(X_tr, mu0_tr, mu1_tr, rng, alpha=alpha)
                if T_tr.sum() < 5 or (1 - T_tr).sum() < 5:
                    continue

                # Tune LightGBM on THIS scenario's data — best model for this exact setting
                ctrl_mask  = T_tr == 0
                trt_mask   = T_tr == 1
                X_with_T   = np.column_stack([X_tr, T_tr])
                s_s        = _tune_reg(X_with_T,         Y_tr, T_tr, LGBM_GRID, N_ITER)
                r_nuisance = _tune_reg(X_tr,             Y_tr, T_tr, LGBM_GRID, N_ITER)
                s_cls      = _tune_cls(X_tr,             T_tr,       LGBM_GRID, N_ITER)
                s_ctrl     = _tune_reg_arm(X_tr[ctrl_mask], Y_tr[ctrl_mask], LGBM_GRID, N_ITER)
                s_trt      = _tune_reg_arm(X_tr[trt_mask],  Y_tr[trt_mask],  LGBM_GRID, N_ITER)

                for mname in models:
                    try:
                        t0 = time.time()
                        if mname == 'CausalPFN':
                            learner = CATEEstimator(device=_cpfn_device, verbose=False)
                            learner.fit(
                                np.asarray(X_tr, dtype=np.float32),
                                T_tr.astype(np.float32),
                                Y_tr.astype(np.float32))
                            te = np.asarray(
                                learner.estimate_cate(np.asarray(X_te, dtype=np.float32))
                            ).reshape(-1)
                        else:
                            learner = _make_learner(mname, s_s, r_nuisance, s_cls, s_ctrl, s_trt)
                            if mname == 'CausalForest':
                                learner.tune(Y_tr, T_tr, X=X_tr)   # original calls .tune() before .fit()
                            learner.fit(Y_tr, T_tr, X=X_tr)
                            te = learner.effect(X_te)
                        elapsed = time.time() - t0
                        timings[mname].append(elapsed)
                        print(f"  cnf α={alpha:<5.1f} | {mname:20s} | {elapsed:.3f}s")
                        rows.append(dict(
                            axis='confounding', param=alpha, model=mname,
                            rep=rep_id, seed_rep=seed_rep,
                            pehe=calculate_pehe(te, true_ite),
                            ate_error=calculate_ate_error(te, true_ite),
                        ))
                    except Exception as e:
                        print(f"\n[confounding] {mname} rep={rep_id} alpha={alpha}: {e}")


    # ── Timing summary ────────────────────────────────────────────────────────
    print("\n── Timing summary (mean ± std seconds per fit) ──────────────────")
    for mname in models:
        t_arr = timings[mname]
        if t_arr:
            print(f"  {mname:20s}: {np.mean(t_arr):.3f}s ± {np.std(t_arr):.3f}s  (n={len(t_arr)})")

    return pd.DataFrame(rows)




In [8]:
# Distinct colors for standalone causal models in plots
_MODEL_COLORS = {
    'CausalPFN':    '#d62728',  # red
    'CausalForest': '#9467bd',  # purple
}

In [9]:
# ── Visualization ─────────────────────────────────────────────────────────────
def plot_sensitivity(df, save_path='sensitivity_analysis_acic.png'):
    """
    2×2 grid: rows = metrics (PEHE, ATE error), cols = axes (imbalance, confounding).
    CausalPFN and CausalForest are plotted in distinct fixed colors.
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    axis_meta = {
        'imbalance':   ('P(T=1)  [0.5 = balanced]',   'Imbalance  [T ⊥ X,  P(T) varies]'),
        'confounding': ('α  [0 = no confounding]',     'Confounding  [P(T) ≈ 0.5,  α varies]'),
    }

    for col, axis_key in enumerate(['imbalance', 'confounding']):
        df_ax  = df[df['axis'] == axis_key]
        xlabel, title_suffix = axis_meta[axis_key]

        for row, metric in enumerate(['pehe', 'ate_error']):
            ax = axes[row][col]
            for model in sorted(df_ax['model'].unique()):
                grp   = df_ax[df_ax['model'] == model].groupby('param')[metric]
                means = grp.mean()
                sems  = grp.sem()
                color = _MODEL_COLORS.get(model, None)
                lw    = 2.5 if model in _MODEL_COLORS else 1.8
                ls    = '--' if model in _MODEL_COLORS else '-'
                ax.errorbar(means.index, means.values, yerr=sems.values,
                            marker='o', capsize=4, linewidth=lw, linestyle=ls,
                            color=color, label=model)

            ax.set_xlabel(xlabel)
            ax.set_ylabel(metric.upper())
            ax.set_title(f'{metric.upper()} — {title_suffix}')
            ax.legend(framealpha=0.9)
            ax.grid(True, alpha=0.3)

    plt.suptitle('Sensitivity Analysis: Imbalance vs. Confounding (ACIC 2016)', y=1.01, fontsize=13)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Figure saved → {save_path}")

In [ ]:
# ── Run ────────────────────────────────────────────────────────────────────────
# n_reps=1: each ACIC 2016 instance is evaluated with 1 independent T-assignment
# per scenario → 10 × 1 = 10 samples per scenario point.
print("Running sensitivity analysis (n_reps=1 per instance per scenario)...")
sens_df = run_sensitivity_analysis(processed_datasets, models=SENS_MODELS, n_reps=1, seed=42)
sens_df.to_csv('sensitivity_results_ihdp0404.csv', index=False)
print(f"Saved → sensitivity_results_acic.csv  ({len(sens_df)} rows)\n")

# ── Summary table ──────────────────────────────────────────────────────────────
print("── Imbalance axis (mean over all instances & seeds) ─────────────────")
print(sens_df[sens_df['axis'] == 'imbalance']
      .groupby(['param', 'model'])[['pehe', 'ate_error']]
      .mean().round(4).to_string())

print("\n── Confounding axis ────────────────────────────────────────────────────")
print(sens_df[sens_df['axis'] == 'confounding']
      .groupby(['param', 'model'])[['pehe', 'ate_error']]
      .mean().round(4).to_string())

# ── Plot ───────────────────────────────────────────────────────────────────────
plot_sensitivity(sens_df)

Running sensitivity analysis (n_reps=1 per instance per scenario)...

── Rep   1/5 ────────────────────────────────


/opt/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/miniconda3/lib/python3.13

  LightGBM tuning: 280.7s
  imb p=0.50 | S-LinearRegression   | 0.004s
  imb p=0.50 | S-LightGBM           | 2.153s
  imb p=0.50 | S-TabPFN             | 6.573s
  imb p=0.50 | S-TabICL             | 9.812s
  imb p=0.50 | T-LinearRegression   | 0.010s
  imb p=0.50 | T-LightGBM           | 3.128s
  imb p=0.50 | T-TabPFN             | 5.308s
  imb p=0.50 | T-TabICL             | 9.982s
  imb p=0.50 | X-LinearRegression   | 0.024s


/opt/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/miniconda3/lib/python3.13

  imb p=0.50 | X-LightGBM           | 55.137s
  imb p=0.50 | X-TabPFN             | 20.636s
  imb p=0.50 | X-TabICL             | 31.938s
  imb p=0.50 | R-LinearRegression   | 0.057s


...

  cnf α=1.0   | R-TabPFN             | 123.504s


/opt/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/miniconda3/lib/python3.13

  cnf α=1.0   | R-TabICL             | 213.959s
  cnf α=1.0   | DR-LinearRegression  | 0.024s


/opt/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/miniconda3/lib/python3.13

  cnf α=1.0   | DR-LightGBM          | 49.331s
